# Домашнее задание: Pydantic


## Важно!

- При выполнении задания используем точные типы (`EmailStr`, `HttpUrl`, `SecretStr`, `Decimal`, конкретные `Enum`).
- Придерживаемся принципа разделения валидаций: проверка поля — в `field_validator`, сквозные зависимости — в `model_validator`


## Задача 1. Профиль пользователя (валидация полей)

Постройте модель профиля пользователя для внутренней CRM:

**Требования**
1. Обязательные поля: `id: UUID`, `email: EmailStr`, `name: str`.
2. Опциональные поля: `website: HttpUrl | None`, `bio: str | None`.
3. Пароль хранится как `SecretStr`, должен быть не короче 8 символов.
4. Имя (`name`) нормализуйте: тримминг + одна пробельная последовательность между словами + первая буква каждого слова заглавная.
5. Если указан `website`, домен сайта не должен совпадать с доменом `email` (смысл: личный сайт != корпоративная почта).

Подсказки: используйте `field_validator` для нормализации и локальных проверок; и `model_validator(mode="after")` для проверки зависимости `email` ↔ `website`.


In [1]:
!pip install -U pydantic[email,timezone] -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.4/463.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.4 which is incompatible.


In [2]:
from typing import Optional
from pydantic import BaseModel, Field, EmailStr, HttpUrl, SecretStr
from pydantic import field_validator, model_validator
from uuid import UUID

class UserProfile(BaseModel):
    id: UUID
    email: EmailStr
    name: str
    password: SecretStr
    website: Optional[HttpUrl] = None
    bio: Optional[str] = None

    # нормализация имени
    @field_validator("name")
    @classmethod
    def normalize_name(cls, v: str) -> str:
        words = v.split()
        return " ".join(word.capitalize() for word in words)

    # проверка длины пароля
    @field_validator("password")
    @classmethod
    def password_strength(cls, v: SecretStr) -> SecretStr:
        raw = v.get_secret_value()
        if len(raw) < 8:
            raise ValueError("Password must be at least 8 characters long")
        return v

    # сквозная проверка доменов email/website
    @model_validator(mode="after")
    def check_domains(self):
        if self.website is None:
            return self

        email_domain = str(self.email).split("@")[-1].lower()
        host = (self.website.host or "").lower()

        if host.startswith("www."):
            host = host[4:]

        if host == email_domain:
            raise ValueError("Website domain must differ from email domain")

        return self


## Задача 2. Валидация функции заказа (`@validate_call`)

Реализуйте функцию `place_order`, которая принимает:
- `user_id: UUID`
- `sku: str` (артикул, только заглавные буквы/цифры, длина 3–12)
- `quantity: int` (>0)
- `price: Decimal` (>= 0), округляется банковским методом до 2 знаков

Функция должна возвращать словарь с ключами: `user_id`, `sku`, `quantity`, `price`, `amount` (quantity × price).

Используйте `@validate_call` и локальные проверки через обычный код (или вспомогательные валидаторы `TypeAdapter` не используем).


In [3]:
from pydantic import validate_call
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
import re

SKU_PATTERN = re.compile(r"^[A-Z0-9]{3,12}$")

# реализуем функцию с @validate_call
@validate_call
def place_order(user_id: UUID, sku: str, quantity: int, price: Decimal) -> dict:
    sku_norm = sku.strip().upper()
    if not SKU_PATTERN.fullmatch(sku_norm):
        raise ValueError("Invalid SKU format")

    if quantity <= 0:
        raise ValueError("quantity must be > 0")

    if price < 0:
        raise ValueError("price must be >= 0")

    price = price.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)
    amount = (price * quantity).quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

    return {
        "user_id": user_id,
        "sku": sku_norm,
        "quantity": quantity,
        "price": price,
        "amount": amount,
    }


## Задача 3. Модель заказа с бизнес-правилами

Смоделируйте заказ в магазине цифровых товаров.

**Требования**
- `OrderStatus: Enum` со значениями `new`, `paid`, `delivered`, `canceled`.
- Модель `OrderItem`:
  - `sku: str` как в задаче 2
  - `qty: int` (>0)
  - `unit_price: Decimal` (>=0) округление до 2 знаков
- Модель `Order`:
  - `id: UUID`
  - `user_email: EmailStr`
  - `items: list[OrderItem]` (не пустой)
  - `status: OrderStatus = 'new'`
  - `created_at: datetime` (по умолчанию `datetime.utcnow`)
  - Расчитанное поле `total: Decimal` — сумма по всем позициям
  - В `model_validator(mode="after")` запретите переход в `paid`/`delivered` при `total == 0` и запретите пустые корзины.

**Важно:** используйте только инструменты `pydantic` и стандартную библиотеку.


In [4]:
import re
from pydantic import BaseModel, EmailStr, field_validator, model_validator, Field
from typing import List
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
from datetime import datetime
from enum import Enum

SKU_RE = re.compile(r"^[A-Z0-9]{3,12}$")


class OrderStatus(str, Enum):
    NEW = "new"
    PAID = "paid"
    DELIVERED = "delivered"
    CANCELED = "canceled"


class OrderItem(BaseModel):
    sku: str
    qty: int
    unit_price: Decimal

    @field_validator("sku")
    @classmethod
    def sku_format(cls, v: str) -> str:
        v = v.strip().upper()
        if not SKU_RE.fullmatch(v):
            raise ValueError("Invalid SKU format")
        return v

    @field_validator("qty")
    @classmethod
    def qty_positive(cls, v: int) -> int:
        if v <= 0:
            raise ValueError("qty must be > 0")
        return v

    @field_validator("unit_price")
    @classmethod
    def price_non_negative(cls, v: Decimal) -> Decimal:
        if v < 0:
            raise ValueError("unit_price must be >= 0")
        return v.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)


class Order(BaseModel):
    id: UUID
    user_email: EmailStr
    items: List[OrderItem]
    status: OrderStatus = OrderStatus.NEW
    created_at: datetime = Field(default_factory=datetime.utcnow)
    total: Decimal = Decimal("0.00")

    @model_validator(mode="after")
    def check_business_rules(self):
        if not self.items:
            raise ValueError("Order must have at least one item")

        total = sum(
            (item.unit_price * item.qty for item in self.items),
            Decimal("0.00"),
        ).quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)
        self.total = total

        if self.total == 0 and self.status in {OrderStatus.PAID, OrderStatus.DELIVERED}:
            raise ValueError("Cannot set paid/delivered status for zero-total order")

        return self


## Задача 4. Конфигурация приложения (`BaseSettings`)

Опишите настройки подключения к внешнему API:

- `APISettings(BaseSettings)` с полями:
  - `base_url: HttpUrl`
  - `token: SecretStr`
  - `timeout_sec: int = 5` (1–60)
  - `retries: int = 2` (0–10)
- Используйте `model_config = ConfigDict(env_prefix="API_", env_file=".env", extra="ignore")`
- Проверьте, что значения корректно читаются из переменных окружения.

В тесте ниже среда заполняется вручную.


In [5]:
import os
from pydantic_settings import BaseSettings
from pydantic import ConfigDict, SecretStr, HttpUrl, field_validator

class APISettings(BaseSettings):
    base_url: HttpUrl
    token: SecretStr
    timeout_sec: int = 5
    retries: int = 2

    # пример проверки диапазона для timeout_sec / retries
    @field_validator("timeout_sec", "retries")
    @classmethod
    def check_ranges(cls, v: int, info):
        field_name = getattr(info, "field_name", "")
        if field_name == "timeout_sec":
            if not 1 <= v <= 60:
                raise ValueError("timeout_sec must be between 1 and 60")
        elif field_name == "retries":
            if not 0 <= v <= 10:
                raise ValueError("retries must be between 0 and 10")
        return v

    model_config = ConfigDict(
        env_prefix="API_",
        env_file=".env",
        extra="ignore",
    )


## Задача 5. Извлечение из ORM (`from_attributes=True`)

Создайте простую SQLAlchemy-модель `SAUser(id, email, is_active)` (in-memory, без БД) и соответствующую модель Pydantic:

- Pydantic-модель `UserOut` с полями `id: UUID`, `email: EmailStr`, `is_active: bool`.
- Включите поддержку `from_attributes` в `model_config`.
- Создайте инстанс `SAUser` и провалидируйте его через `UserOut.model_validate(sa_user_instance)`.

Проверьте, что преобразование сработало.


In [6]:
from typing import Optional
from sqlalchemy import Column, String, Boolean
from sqlalchemy.orm import declarative_base
from uuid import uuid4, UUID
from pydantic import BaseModel, EmailStr, ConfigDict

Base = declarative_base()

class SAUser(Base):
    __tablename__ = "users"
    id = Column(String, primary_key=True, default=lambda: str(uuid4()))
    email = Column(String, nullable=False)
    is_active = Column(Boolean, default=True)

    def __init__(self, email: str, is_active: bool = True):
        self.id = str(uuid4())
        self.email = email
        self.is_active = is_active

class UserOut(BaseModel):
    id: UUID
    email: EmailStr
    is_active: bool

    model_config = ConfigDict(from_attributes=True)


## Задача 6. JSON Schema и дружелюбные ошибки

1. Для модели из задачи 3 сгенерируйте JSON Schema (метод `model_json_schema`) и запишите его в переменную `ORDER_SCHEMA`.
2. Реализуйте функцию `safe_create_order(data: dict) -> tuple[bool, str]`, которая:
   - пытается создать `Order` из входного `dict`,
   - при успехе возвращает `(True, "<total=...>")`,
   - при ошибке возвращает `(False, "<короткое сообщение об ошибке>")` без стек-трейса.

Не используйте сторонние библиотеки.


In [7]:
from pydantic import ValidationError

# Используем модели из задачи 3: OrderStatus, OrderItem, Order

ORDER_SCHEMA = Order.model_json_schema()

def safe_create_order(data: dict) -> tuple[bool, str]:
    try:
        order = Order.model_validate(data)
    except ValidationError as e:
        errors = e.errors()
        if errors:
            msg = errors[0].get("msg", str(e))
        else:
            msg = str(e)
        return False, msg
    except Exception as e:
        return False, str(e)
    else:
        return True, f"total={order.total}"
